# 🌍 Global Terrorism Analysis — End-to-End Data Science Project

---

## Research Question

> **How can historical terrorism data be analyzed to uncover patterns in attack types, locations, targets, and perpetrators, and how effectively can these insights be used to assess terrorism risk across regions and time periods?**

---

## Motivating Questions

| # | Question |
|---|---------|
| 1 | Where are global terrorism threats most concentrated? |
| 2 | How do patterns change across countries and over time? |
| 3 | What insights can past data reveal about future risk? |
| 4 | Which patterns are hidden within raw incident data? |
| 5 | Can data science identify high-risk regions and emerging trends? |

---

## Notebook Structure

| Section | Focus |
|---------|-------|
| **1. Raw Data** | Source tables, shapes, columns |
| **2. Data Cleaning** | Profiling, issues found, steps taken, cleaned outputs |
| **3. Master Table** | How all sources were merged and what the final table contains |
| **4. Descriptive Analysis** | What happened — summary statistics, trends, distributions |
| **5. Diagnostic Analysis** | Why it happened — correlation, regression, hypothesis tests, root cause |
| **6. Predictive Analysis** | What will happen — forecasting and risk classification |
| **7. Prescriptive Analysis** | What to do — rule engine, resource-allocation optimisation |


---
## Section 1 — Raw Data Sources

Four datasets were collected from authoritative public repositories. All raw files are stored under `data/raw/` and treated as **immutable** — they are never edited in place.

| Dataset | File | Source | Records | Key Columns |
|---------|------|--------|---------|-------------|
| **GTD** (Global Terrorism Database) | `archive/globalterrorismdb_0718dist.csv` | Kaggle / START-UMD | 181,691 rows × 8 cols | year, country, region, attack type, target, perpetrator, kills, wounds |
| **UCDP Organised Violence** | `organizedviolencecy_v25_1.csv` | UCDP Uppsala | 6,936 rows × 4 cols | country, year, region, cumulative conflict deaths |
| **OWID Population** | `population-with-un-projections.csv` | Our World in Data / UN WPP 2024 | ~75,000 rows × 3 cols | entity, year, population |
| **Maddison GDP per Capita** | `gdp-per-capita-maddison-project-database.csv` | Our World in Data / Maddison Project | 21,586 rows × 3 cols | entity, year, GDP per capita |


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
import warnings, os, sys
warnings.filterwarnings("ignore")

# Make notebook output robust on Windows terminals that default to cp1252.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy import stats

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# Resolve paths (works from any working directory)
try:
    REPO = Path(__file__).resolve().parents[0]
except NameError:
    REPO = Path(".").resolve()
    if not (REPO / "data").exists():
        # Try relative to common notebook launch point
        for candidate in [Path("data_science_project-"), Path(".")]:
            if (candidate / "data").exists():
                REPO = candidate.resolve()
                break

DATA_RAW        = REPO / "data" / "raw"
DATA_INTERIM    = REPO / "data" / "interim"
DATA_FINAL      = REPO / "data" / "final"
ANALYSIS        = REPO / "analysis"

print("Project root:", REPO)
print("Raw data:    ", DATA_RAW)
print("Interim:     ", DATA_INTERIM)
print("Final:       ", DATA_FINAL)


## Section 0 — Full Project Folder Map and README Digest

This section makes the notebook self-contained as project documentation:

- It lists all major folders and Python entrypoints.
- It surfaces every `README.md`/`readme.md` under the repository.
- It provides enough context so this notebook can be read as a single guide across `data/`, `src/`, `analysis/`, and `tests/`.

In [ ]:
# ── 0.1 Project inventory (folders + Python files) ─────────────────
from collections import defaultdict

major_folders = ["data", "src", "analysis", "tests"]
for folder in major_folders:
    p = REPO / folder
    exists = p.exists()
    py_files = sorted(p.rglob("*.py")) if exists else []
    nb_files = sorted(p.rglob("*.ipynb")) if exists else []
    print(f"\n[{folder}] exists={exists}")
    print(f"  python files:   {len(py_files)}")
    print(f"  notebooks:      {len(nb_files)}")

    if py_files:
        print("  sample entrypoints:")
        for fp in py_files[:12]:
            print("   -", fp.relative_to(REPO).as_posix())
        if len(py_files) > 12:
            print(f"   ... +{len(py_files)-12} more")

all_py = sorted(REPO.rglob("*.py"))
print(f"\nTotal Python files in repo: {len(all_py)}")

In [ ]:
# ── 0.2 README digest for all project folders ──────────────────────
readme_paths = sorted(
    [p for p in REPO.rglob("README*.md")] + [p for p in REPO.rglob("readme*.md")],
    key=lambda p: p.as_posix().lower(),
)

# de-duplicate if both patterns catch same file
seen = set()
readme_paths = [p for p in readme_paths if not (p.resolve() in seen or seen.add(p.resolve()))]

print(f"Found {len(readme_paths)} README files")
for rp in readme_paths:
    rel = rp.relative_to(REPO).as_posix()
    print("\n" + "=" * 95)
    print(f"README: {rel}")
    print("=" * 95)

    text = rp.read_text(encoding="utf-8", errors="replace")
    lines = text.splitlines()

    # Keep this section readable in notebook output while still detailed.
    preview = lines[:60]
    print("\n".join(preview) if preview else "(empty file)")
    if len(lines) > 60:
        print(f"\n... truncated: showing first 60 of {len(lines)} lines ...")

In [ ]:
# ── 1.1  GTD (Global Terrorism Database) ───────────────────────────
gtd_raw_path = DATA_INTERIM / "gtd_dataprofile" / "gtd_profile_subset.csv"
gtd_raw = pd.read_csv(gtd_raw_path)

print("GTD Raw — Shape:", gtd_raw.shape)
print("Columns:", list(gtd_raw.columns))
print()
gtd_raw.head(10)


In [ ]:
# ── 1.2  UCDP Organised Violence ────────────────────────────────
ucdp_raw_path = DATA_INTERIM / "conflict_dataprofile" / "conflict_profile_subset.csv"
ucdp_raw = pd.read_csv(ucdp_raw_path)

print("UCDP Raw — Shape:", ucdp_raw.shape)
print("Columns:", list(ucdp_raw.columns))
print("Year range:", ucdp_raw["year_cy"].min(), "–", ucdp_raw["year_cy"].max())
print()
ucdp_raw.head(10)


In [ ]:
# ── 1.3  Population (OWID) ──────────────────────────────────────
pop_raw_path = DATA_INTERIM / "population+gdp_dataprofile" / "population_profile_subset.csv"
pop_raw = pd.read_csv(pop_raw_path)

print("Population Raw — Shape:", pop_raw.shape)
print("Columns:", list(pop_raw.columns))
print()
pop_raw.head(10)


In [ ]:
# ── 1.4  GDP per Capita (Maddison Project) ──────────────────────
gdp_raw_path = DATA_INTERIM / "population+gdp_dataprofile" / "gdp_profile_subset.csv"
gdp_raw = pd.read_csv(gdp_raw_path)

print("GDP Raw — Shape:", gdp_raw.shape)
print("Columns:", list(gdp_raw.columns))
print()
gdp_raw.head(10)


---
## Section 2 — Data Profiling & Cleaning

For each dataset the pipeline follows a two-step process:

1. **Profiling** — understand shape, types, missingness, duplicates, and distribution
2. **Cleaning** — apply targeted fixes, then export to `data/interim/`

All cleaned files are stored separately from the raw files so the originals remain untouched.


### 2.1  GTD — Profiling

The GTD is the largest table (181,691 events, 8 columns). The profiler examined:
- Column names, dtypes, and unique value counts
- Missing value counts and percentages
- Duplicate rows
- Distribution of `nkill` and `nwound`


In [ ]:
# ── 2.1 GTD Profiling ───────────────────────────────────────────
print("=" * 65)
print("GTD DATA PROFILING REPORT")
print("=" * 65)

# Re-read from profile subset (first 5000 rows used by profiler for speed)
gtd_profile = pd.read_csv(DATA_INTERIM / "gtd_dataprofile" / "gtd_profile_subset.csv")

print(f"\nShape: {gtd_profile.shape}")
print("\nColumn dtypes:")
print(gtd_profile.dtypes.to_string())

print("\nMissing values (count and %):")
missing = gtd_profile.isnull().sum()
pct     = (missing / len(gtd_profile) * 100).round(2)
mv = pd.DataFrame({"Missing Count": missing, "Missing %": pct})
print(mv[mv["Missing Count"] > 0].to_string())

print(f"\nDuplicate rows: {gtd_profile.duplicated().sum()}")

print("\nUnique values per column:")
print(gtd_profile.nunique().to_string())

print("\nDescriptive statistics for numeric columns:")
print(gtd_profile[["nkill","nwound"]].describe().round(2).to_string())


: 

#### GTD Profiling Findings

| Issue Found | Magnitude | Severity |
|-------------|-----------|----------|
| Missing `nkill` (fatalities) | 5.68% of rows | Medium — must be handled |
| Missing `nwound` (injuries) | 8.98% of rows | Medium — must be handled |
| Duplicate rows | 76,991 (42%) | High — caused by multi-row event encoding in original GTD |
| `Perpetrator = "Unknown"` or empty | ~40% | Medium — normalise to N/A |
| Column names use raw GTD codes | — | Low — cosmetic rename needed |


### 2.2  GTD — Cleaning Steps

The `src/gtd_cleaning_pipeline.py` script applied the following transformations to produce `data/interim/gtd_dataclean/gtd_cleaned.csv` (event-level) and `gtd_country_year.csv` (aggregated).

**Step-by-step:**

1. **Column selection** — Only 8 of the 135 GTD columns are relevant: `iyear`, `country_txt`, `region_txt`, `attacktype1_txt`, `targtype1_txt`, `gname`, `nkill`, `nwound`.
2. **Column rename** — Raw GTD codes → human-readable names: `iyear→Year`, `country_txt→Country`, `nkill→Fatalities`, etc.
3. **Type coercion** — `Year`, `Fatalities`, `Injuries` cast to numeric; `coerce` mode replaces non-numeric garbage with NaN.
4. **Drop rows with missing Country or Year** — These records cannot be located in space-time and are unusable.
5. **Year → int64** — After NaN-drop, safe to convert.
6. **Fill missing Fatalities / Injuries with 0** — Conservative approach: a missing value means "unreported", not "non-zero"; filling with 0 is standard GTD practice.
7. **Normalise Perpetrator field** — Rows where `gname == "Unknown"` or empty string → `pd.NA → "N/A"`.
8. **Country-year aggregation** — Events are grouped by `(Country, Year)` and aggregated:
   - `Num_Attacks = count()`
   - `Fatalities = sum()`, `Injuries = sum()`
   - `Attack_Type_Mode`, `Target_Type_Mode`, `Perpetrator_Mode` = modal (most common) value per group using a safe mode function that handles empty groups.


In [ ]:
# ── 2.2 GTD Cleaned — Event Level ───────────────────────────────
gtd_clean = pd.read_csv(DATA_INTERIM / "gtd_dataclean" / "gtd_cleaned.csv")

print("GTD Cleaned (event-level) — Shape:", gtd_clean.shape)
print("\nSample rows:")
display_df = gtd_clean.head(10)
print(display_df.to_string(index=False))

print("\nMissing values after cleaning:")
mv_clean = gtd_clean.isnull().sum()
print(mv_clean[mv_clean > 0].to_string() if mv_clean.any() else "  None ✓")

print("\nYear range:", gtd_clean["Year"].min(), "–", gtd_clean["Year"].max())
print("Countries:", gtd_clean["Country"].nunique())
print("Unique Attack Types:", gtd_clean["Attack_Type"].unique())


In [ ]:
# ── 2.2 GTD Country-Year Aggregated Table ────────────────────────
gtd_cy = pd.read_csv(DATA_INTERIM / "gtd_dataclean" / "gtd_country_year.csv")

print("GTD Country-Year — Shape:", gtd_cy.shape)
print("\nSample rows:")
print(gtd_cy.head(10).to_string(index=False))

print("\nDescriptive stats (Num_Attacks, Fatalities, Injuries):")
print(gtd_cy[["Num_Attacks","Fatalities","Injuries"]].describe().round(2).to_string())


### 2.3  UCDP Organised Violence — Profiling & Cleaning

**Profiling findings:**
- 6,936 rows, 4 columns — no missing values, no duplicates
- Year range 1989–2024
- One numeric column: `cumulative_total_deaths_in_orgvio_best_cy` (min 0, max 772,463)
- Country names use UCDP naming convention, which must later be harmonised with GTD names

**Cleaning steps:**
1. **Column selection & rename** — `country_cy→Country`, `year_cy→Year`, `region_cy→Region`, deaths column → `Conflict_Intensity_deaths`
2. **Type coercion** — Country/Region to string (strip whitespace); Year to numeric; deaths to numeric
3. **Drop rows with missing Country or Year**
4. **Fill missing `Conflict_Intensity_deaths` with 0**
5. **Create binary `Conflict` flag** — `1` if `Conflict_Intensity_deaths > 0`, else `0`
6. **Create ordinal `Conflict_Level`** — 0–10 scale based on death count thresholds (0 → 0, 1–10 → 1, 11–25 → 2, ..., >5000 → 10). This converts a raw death count into a comparable severity scale.
7. **Deduplicate by Country-Year** — Modal region, summed deaths, max conflict flag/level.


In [ ]:
# ── 2.3 UCDP Cleaned ────────────────────────────────────────────
ucdp_clean = pd.read_csv(DATA_INTERIM / "conflict_dataclean" / "conflict_cleaned.csv")

print("UCDP Cleaned — Shape:", ucdp_clean.shape)
print("\nSample rows:")
print(ucdp_clean.head(10).to_string(index=False))

print("\nConflict_Level distribution:")
print(ucdp_clean["Conflict_Level"].value_counts().sort_index().to_string())

print("\nConflict flag (0=peace, 1=active conflict):")
print(ucdp_clean["Conflict"].value_counts().to_string())

print("\nMissing values after cleaning:")
mv = ucdp_clean.isnull().sum()
print(mv[mv > 0].to_string() if mv.any() else "  None ✓")


### 2.4  Population — Profiling & Cleaning

**Profiling findings:**
- The raw file contains both genuine countries and regional aggregates (World, Africa, income groups, etc.)
- Population values span 1800–2100 (includes projections)
- Some non-country rows have `OWID_*` codes in the `Code` column

**Cleaning steps:**
1. **Column selection** — Entity, Year, Population column (name varies by export version)
2. **Rename** — `Entity→Country`, dynamic population column → `Population`
3. **Remove non-country rows** — Use `Code` column: keep only ISO-like 3-letter codes (`[A-Z]{3}`), exclude `OWID_*` prefixes. This removes World, continents, income groups.
4. **Type coercion** — Year → int64, Population → float64
5. **Drop rows with missing Country, Year, or Population**
6. **Deduplicate** — Keep first occurrence per `(Country, Year)` pair


In [ ]:
# ── 2.4 Population Cleaned ──────────────────────────────────────
pop_clean = pd.read_csv(DATA_INTERIM / "population+gdp_dataclean" / "population_cleaned.csv")

print("Population Cleaned — Shape:", pop_clean.shape)
print("\nSample rows:")
print(pop_clean.head(8).to_string(index=False))

print("\nYear range:", pop_clean["Year"].min(), "–", pop_clean["Year"].max())
print("Unique countries:", pop_clean["Country"].nunique())
print("\nPopulation summary:")
print(pop_clean["Population"].describe().apply(lambda x: f"{x:,.0f}").to_string())

print("\nMissing values:")
mv = pop_clean.isnull().sum()
print(mv[mv > 0].to_string() if mv.any() else "  None ✓")


### 2.5  GDP per Capita — Profiling & Cleaning

**Profiling findings:**
- 21,586 rows; columns: Entity, Code, Year, `GDP per capita`
- No missing values in the raw profiling subset
- Year range: 1 to 2022 (historical data going back centuries)
- 380 non-country entries (regional aggregates, income groups)

**Cleaning steps:**
1. **Select columns** — Entity, Code, Year, `GDP per capita`
2. **Rename** — `Entity→Country`, `GDP per capita→GDP_per_capita`
3. **Strip whitespace** from Country
4. **Remove non-country rows** — Same ISO-code filter as population (`[A-Z]{3}`, no `OWID_*`)
5. **Type coercion** — Year → int64, GDP_per_capita → float64
6. **Drop rows with missing Country or Year**
7. **Deduplicate** by `(Country, Year)`, keep first
8. **Interpolate missing GDP** — Linear interpolation within each country group across years (`limit_direction="both"`). This fills in gaps in historical records that would otherwise leave large NaN regions in the master table.


In [ ]:
# ── 2.5 GDP Cleaned ─────────────────────────────────────────────
gdp_clean = pd.read_csv(DATA_INTERIM / "population+gdp_dataclean" / "gdp_cleaned.csv")

print("GDP Cleaned — Shape:", gdp_clean.shape)
print("\nSample rows:")
print(gdp_clean.head(8).to_string(index=False))

print("\nYear range:", gdp_clean["Year"].min(), "–", gdp_clean["Year"].max())
print("Unique countries:", gdp_clean["Country"].nunique())
print("\nGDP per capita summary:")
print(gdp_clean["GDP_per_capita"].describe().apply(lambda x: f"{x:,.2f}").to_string())
print("\nMissing values:")
mv = gdp_clean.isnull().sum()
print(mv[mv > 0].to_string() if mv.any() else "  None ✓")


### 2.6  Cleaning Summary

| Table | Raw Shape | Cleaned Shape | Key Issues Resolved |
|-------|-----------|---------------|---------------------|
| GTD (events) | 181,691 × 8 | ~104,700 × 8 | Duplicates removed, fatalities/injuries filled with 0, perpetrator normalised, columns renamed |
| GTD (country-year) | — | 3,544 × 9 | Aggregated: count → Num_Attacks, sum → Fatalities/Injuries, mode → Attack_Type/Target/Perpetrator |
| UCDP | 6,936 × 4 | 6,936 × 6 | Renamed, binary Conflict flag + ordinal Conflict_Level derived, deduplicated |
| Population | ~75,000 × 3 | ~9,400 × 3 | Non-country aggregates removed via ISO code filter, types coerced |
| GDP per capita | 21,586 × 3 | ~9,500 × 3 | Non-country aggregates removed, linear interpolation for gaps |


---
## Section 3 — Building the Master Table

### 3.1  Merge Strategy

The `src/final_table_pipeline.py` script constructs a single analytics-ready table called `master_country_year.csv`. The design principle is **GTD-anchored left-joins**: the GTD country-year aggregation acts as the base table, and all other datasets are joined onto it.

**Steps:**

1. **Load GTD country-year** (`data/interim/gtd_dataclean/gtd_country_year.csv`) as the primary left side — only rows with at least one recorded terrorism incident enter the master table.
2. **Country name harmonisation** — GTD and UCDP use different country naming conventions (e.g. "United States" vs "USA", "Ivory Coast" vs "Côte d'Ivoire"). A `country_name_mapping.csv` lookup applies standardised names before merging. A `country_mapping_applied` flag records which rows were remapped.
3. **Left-join UCDP** on `(Country, Year)` — adds `Conflict`, `Conflict_Level`, `Conflict_Intensity_deaths`, `Conflict_Region`. Rows with no match get 0/NaN.
4. **Left-join Population** on `(Country, Year)` — adds `Population`.
5. **Left-join GDP** on `(Country, Year)` — adds `GDP_per_capita`.
6. **Data-presence flags** — Binary columns `has_conflict_data`, `has_population_data`, `has_gdp_data` mark whether each row successfully matched in each join. These are useful for filtering or weighting analyses.
7. **Final column selection & type enforcement** — All string columns stripped; numeric columns cast; schema validated against `final_table_schema.json`.
8. **Export** — CSV and Parquet under `data/final/`.

### 3.2  Schema

| Column | Type | Description |
|--------|------|-------------|
| Country | str | Country name (GTD standard) |
| Year | int64 | Calendar year |
| Region | str | World region (GTD categorisation, 12 regions) |
| Num_Attacks | int64 | Count of terrorism incidents that year |
| Fatalities | int64 | Sum of deaths across all incidents |
| Injuries | int64 | Sum of injuries across all incidents |
| Attack_Type_Mode | str | Most common attack type that year |
| Target_Type_Mode | str | Most common target type that year |
| Perpetrator_Mode | str | Most common perpetrating group that year |
| Conflict | int64 | 0/1 — UCDP active conflict flag |
| Conflict_Level | int64 | 0–10 ordinal conflict severity |
| Conflict_Intensity_deaths | int64 | UCDP cumulative conflict deaths |
| Conflict_Region | str | UCDP region name |
| Population | float64 | Annual population |
| GDP_per_capita | float64 | GDP per capita (2011 international $) |
| has_conflict_data | int64 | 1 if UCDP row matched |
| has_population_data | int64 | 1 if population row matched |
| has_gdp_data | int64 | 1 if GDP row matched |
| country_mapping_applied | int64 | 1 if country name was harmonised |


In [ ]:
# ── 3.1 Load & Inspect Master Table ────────────────────────────
master = pd.read_csv(DATA_FINAL / "master_country_year.csv")

print("Master Table Shape:", master.shape)
print("\nColumns:", list(master.columns))
print("\nData types:")
print(master.dtypes.to_string())
print("\nYear range:", master["Year"].min(), "–", master["Year"].max())
print("Unique countries:", master["Country"].nunique())
print("Unique regions:", master["Region"].nunique())
print("Regions:", sorted(master["Region"].dropna().unique()))


In [ ]:
# ── 3.2 Master Table — Sample Rows ─────────────────────────────
print("First 10 rows:")
master.head(10)


In [ ]:
# ── 3.3 Master Table — Statistical Summary ──────────────────────
print("Overall summary statistics:")
master.describe().round(2)


In [ ]:
# ── 3.4 Data Coverage Flags ─────────────────────────────────────
print("Data presence flags (% of rows with matched join):")
flags = ["has_conflict_data", "has_population_data", "has_gdp_data"]
for f in flags:
    pct = master[f].mean() * 100
    print(f"  {f:30s}: {pct:.1f}%")

print("\nMissing values in final master table:")
mv = master.isnull().sum()
print(mv[mv > 0].to_string() if mv.any() else "  No missing values ✓")

print("\nCountry-year pairs:", len(master))
print("Attack statistics:")
print(f"  Total recorded attacks: {master['Num_Attacks'].sum():,}")
print(f"  Total fatalities:       {master['Fatalities'].sum():,}")
print(f"  Total injuries:         {master['Injuries'].sum():,}")


---
## Section 4 — Descriptive Analysis

**Purpose:** Describe *what* happened. Summarise the distribution of attacks, fatalities, and injuries across regions, countries, and time. Answer: *Where are terrorism threats most concentrated?*

**Methods used:**
- Summary statistics (mean, median, std, min, max)
- Time-series trend analysis
- Top-N rankings by country
- Attack type proportions (pie chart)
- Country-year heatmap
- Rolling aggregation
- Categorical aggregation


### 4.1  Summary Statistics

**What this analysis does:** Computes mean, median, standard deviation, min, and max for `Num_Attacks`, `Fatalities`, and `Injuries` — first globally, then grouped by Country and Region. This provides a high-level quantitative picture of how attacks are distributed.

**Question it answers:** What is the typical scale of terrorism activity per country-year? How much does it vary?


In [ ]:
# ── 4.1 Global Summary Statistics ───────────────────────────────
cols = ["Num_Attacks", "Fatalities", "Injuries"]
summary = master[cols].agg(["mean", "median", "std", "min", "max"])

print("=== Global Summary Statistics (per country-year row) ===")
print(summary.round(2).to_string())
print(f"\nTotal rows (country-year observations): {len(master):,}")

print("\n=== Average Num_Attacks by Region ===")
avg_region = (
    master.groupby("Region", as_index=False)["Num_Attacks"]
    .mean()
    .rename(columns={"Num_Attacks": "Avg_Num_Attacks"})
    .sort_values("Avg_Num_Attacks", ascending=False)
)
print(avg_region.round(2).to_string(index=False))


In [ ]:
# ── 4.1 Top 15 Countries by Mean Annual Attacks ─────────────────
avg_country = (
    master.groupby("Country", as_index=False)["Num_Attacks"]
    .mean()
    .rename(columns={"Num_Attacks": "Avg_Num_Attacks"})
    .sort_values("Avg_Num_Attacks", ascending=False)
    .head(15)
)
print("=== Top 15 Countries by Average Annual Attacks ===")
print(avg_country.round(2).to_string(index=False))


**Output Interpretation:**

- The distribution is **heavily right-skewed**: the median `Num_Attacks` per country-year is just 5, yet the mean is ~48 and the max is 3,933. Most country-years are low-activity; a small number of high-intensity cases dominate the aggregate.
- **South Asia** leads all regions with a mean of ~205 attacks per country-year — more than double the next region (MENA at ~94). This is driven primarily by Afghanistan, Iraq, and Pakistan.
- **Western Europe** and **East Asia** show comparatively low mean activity, though they are not zero.
- The top 15 countries are heavily concentrated in **South Asia, MENA, and Sub-Saharan Africa**.


### 4.2  Time-Series Analysis

**What this analysis does:** Aggregates `Num_Attacks` globally by year to produce an annual total, fits a linear trend, and identifies peak years.

**Question it answers:** How has global terrorism activity changed over time? Are there notable spikes or troughs?


In [ ]:
# ── 4.2 Time Series — Total Attacks by Year ──────────────────────
yearly = master.groupby("Year")["Num_Attacks"].sum().sort_index()

# Linear trend
years_arr = yearly.index.values.astype(float)
vals_arr  = yearly.values.astype(float)
slope, intercept = np.polyfit(years_arr, vals_arr, 1)
trend = slope * years_arr + intercept

print("=== Yearly Total Num_Attacks ===")
print(f"  First year ({yearly.index[0]}): {yearly.iloc[0]:,}")
print(f"  Last  year ({yearly.index[-1]}): {yearly.iloc[-1]:,}")
print(f"  Linear trend slope: {slope:,.2f} attacks / year")
if slope > 0:
    print("  Direction: POSITIVE — higher totals in later years on average")
else:
    print("  Direction: NEGATIVE — lower totals in later years on average")

peak10 = yearly.nlargest(10).sort_index()
print("\n=== Top 10 Peak Years ===")
print(peak10.to_string())

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(yearly.index, yearly.values, linewidth=2, label="Total attacks")
ax.plot(yearly.index, trend, linestyle="--", linewidth=1.5, label=f"Linear trend (slope={slope:+.0f}/yr)")
top5 = yearly.nlargest(5)
ax.scatter(top5.index, top5.values, zorder=5, s=80, label="Top 5 peak years")
for yr, val in top5.items():
    ax.annotate(str(yr), (yr, val), textcoords="offset points", xytext=(4, 4), fontsize=8)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Num_Attacks", fontsize=12)
ax.set_title("Global Total Terrorism Attacks by Year (1970–2017)", fontsize=14)
ax.legend()
fig.tight_layout()
plt.show()
print("Figure: Total attacks over time with linear trend")


**Output Interpretation:**

- Global terrorism activity increased sharply from the mid-2000s onward, driven by the proliferation of conflicts in the Middle East and South Asia.
- The **peak years are clustered in 2011–2016**, coinciding with the rise of ISIS/ISIL, the Syrian Civil War, and the Taliban resurgence in Afghanistan/Pakistan.
- The linear trend slope is **positive (~+60 attacks/year)**, confirming a long-run upward trend across the full 1970–2017 observation window.
- There is a notable **dip around 1998** and a secondary peak in the early 1990s (Latin American insurgencies, post-Cold War instability).


### 4.3  Top Countries — Bar Chart

**What this analysis does:** Ranks all countries by their total cumulative attack count and plots the top 10.

**Question it answers:** Which countries are the most persistent terrorism hotspots in absolute terms?


In [ ]:
# ── 4.3 Top 10 Countries by Total Attacks ────────────────────────
by_country = master.groupby("Country")["Num_Attacks"].sum()
top10 = by_country.nlargest(10).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top10.index.astype(str), top10.values,
               color=plt.cm.Reds(np.linspace(0.4, 0.9, 10)))
ax.set_xlabel("Total Attacks (1970–2017)", fontsize=12)
ax.set_ylabel("Country", fontsize=12)
ax.set_title("Top 10 Countries by Cumulative Terrorist Attacks", fontsize=14)
for bar, val in zip(bars, top10.values):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f"{val:,}", va="center", fontsize=9)
fig.tight_layout()
plt.show()

print("\nTop 10 Countries:")
top10_df = by_country.nlargest(10).reset_index()
top10_df.columns = ["Country", "Total_Attacks"]
print(top10_df.to_string(index=False))


**Output Interpretation:**

- **Iraq** tops the list with the highest cumulative attack count, reflecting the devastating period of 2003–2017 following the US invasion.
- **Afghanistan** and **Pakistan** follow closely — together these three countries account for a disproportionate share of global terrorism incidents.
- **Colombia** and **India** represent long-running insurgencies (FARC/ELN and Naxalites/Kashmir militants respectively) that stretch across decades.
- The concentration of attacks in a small number of countries highlights the **geographic clustering** of terrorism risk.


### 4.4  Attack Type Distribution

**What this analysis does:** Computes the share of total attacks attributed to each attack type mode and visualises as a pie chart.

**Question it answers:** Which attack method dominates globally?


In [ ]:
# ── 4.4 Attack Type Proportions ──────────────────────────────────
by_type = (
    master.groupby("Attack_Type_Mode", as_index=True)["Num_Attacks"]
    .sum()
    .sort_values(ascending=False)
)
total = float(by_type.sum())
pct = (by_type.values / total * 100).round(1)

print("=== Attack Type Distribution ===")
type_df = pd.DataFrame({
    "Attack_Type": by_type.index.astype(str),
    "Total_Attacks": by_type.values,
    "Share_%": pct
})
print(type_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 7))
legend_labels = [f"{name}: {p:g}%" for name, p in zip(by_type.index.astype(str), pct)]
wedges, _ = ax.pie(by_type.values, labels=None,
                   colors=plt.cm.Set3(np.linspace(0, 1, len(by_type))))
ax.legend(wedges, legend_labels, title="Attack Type", loc="center left",
          bbox_to_anchor=(1.02, 0.5), fontsize=9)
ax.set_title("Attack Type Proportions (Share of Total Num_Attacks)", fontsize=13)
ax.axis("equal")
fig.tight_layout()
plt.show()


**Output Interpretation:**

- **Bombing/Explosion** is overwhelmingly the dominant attack type, accounting for the largest share of incidents globally. This reflects both the lethality and the scalability of explosive devices — they can be deployed remotely, require relatively low operational skill, and cause mass casualties.
- **Armed Assault** (direct gun/melee attacks) is the second most common type, prominent in conflict zones with access to small arms.
- **Assassination** and **Facility/Infrastructure Attacks** are significant but smaller categories.
- **Hijacking** is the rarest, having been sharply curtailed by post-9/11 aviation security measures.


### 4.5  Country-Year Heatmap

**What this analysis does:** Creates a pivot table of `Num_Attacks` (Country × Year) and visualises it as a heatmap, sorted by total attack count. This reveals temporal clustering and country-level persistence.

**Question it answers:** Are high-attack countries persistently active, or are spikes isolated events?


In [ ]:
# ── 4.5 Country-Year Heatmap (Top 30 Countries) ──────────────────
pivot = master.pivot_table(
    index="Country", columns="Year", values="Num_Attacks",
    aggfunc="sum", fill_value=0
).sort_index(axis=1)

# Keep top 30 by total
top30_countries = pivot.sum(axis=1).nlargest(30).index
pivot_top30 = pivot.loc[top30_countries].loc[pivot.sum(axis=1).sort_values(ascending=False).head(30).index]

fig, ax = plt.subplots(figsize=(18, 10))
sns.heatmap(pivot_top30, ax=ax, cmap="YlOrRd",
            cbar_kws={"label": "Num_Attacks"},
            xticklabels=5)
ax.set_title("Terrorism Attack Heatmap — Top 30 Countries × Year", fontsize=14)
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Country", fontsize=12)
fig.tight_layout()
plt.show()
print("Heatmap shows attack density; darker cells = more attacks in that country-year.")


**Output Interpretation:**

- The heatmap reveals that high-attack countries tend to show **persistent activity over many years**, not isolated spikes. Iraq, Afghanistan, and Pakistan show sustained dark cells from the 2000s onward.
- Several countries show **temporal transitions**: Colombia is active through the 1980s–2000s but quietens after 2010 (peace process); Iraq is relatively quiet pre-2003 then explodes post-invasion.
- Western European countries show **brief spikes** (e.g., Northern Ireland Troubles in the UK during the 1970s–90s) but generally much lower intensity.


### 4.6  Rolling Aggregation (3-Year Window)

**What this analysis does:** Computes a 3-year rolling mean of global total attacks to smooth year-to-year noise and highlight medium-term trends.

**Question it answers:** What does smoothed terrorism activity reveal about structural trends versus volatility?


In [ ]:
# ── 4.6 Rolling Aggregation — Global 3-Year Rolling Mean ──────────
yearly_series = master.groupby("Year")["Num_Attacks"].sum().sort_index()

rolling3 = yearly_series.rolling(window=3, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(yearly_series.index, yearly_series.values, alpha=0.3, label="Annual total")
ax.plot(rolling3.index, rolling3.values, linewidth=2.5, label="3-year rolling mean")
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Num_Attacks", fontsize=12)
ax.set_title("Global Attacks — Annual Total vs 3-Year Rolling Mean", fontsize=14)
ax.legend()
fig.tight_layout()
plt.show()

print("Rolling aggregation (last 5 years):")
print(pd.DataFrame({"Annual": yearly_series.tail(5), "Rolling_3yr": rolling3.tail(5)}).round(1).to_string())


### 4.7  Categorical Aggregation — Attacks by Region × Year

**What this analysis does:** Aggregates `Num_Attacks` by `(Region, Year)` and plots a stacked area chart to show how different regions contributed to global totals over time.

**Question it answers:** Which regions drove the overall increase in terrorism? Did that change over time?


In [ ]:
# ── 4.7 Categorical Aggregation — Region × Year ──────────────────
region_year = (
    master.groupby(["Year", "Region"])["Num_Attacks"]
    .sum()
    .unstack(fill_value=0)
    .sort_index(axis=1)
)

fig, ax = plt.subplots(figsize=(14, 6))
region_year.plot(kind="area", stacked=True, ax=ax, alpha=0.75,
                 colormap="tab20")
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Num_Attacks", fontsize=12)
ax.set_title("Global Attacks by Region Over Time (Stacked Area)", fontsize=14)
ax.legend(title="Region", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
fig.tight_layout()
plt.show()

print("\nRegion × Year pivot (last 5 years, top regions):")
top_regions = region_year.sum().nlargest(5).index
print(region_year[top_regions].tail(5).to_string())


**Output Interpretation (Rolling & Categorical):**

- The rolling mean confirms the structural upward trend is **not driven by a single outlier year** — the trend persists across multiple windows.
- The stacked area chart shows that **Middle East & North Africa (MENA)** surpassed South Asia as the dominant contributor in the 2013–2016 window, driven by Iraq and Syria.
- **Sub-Saharan Africa** shows a notable rise from 2010 onward, reflecting the emergence of Boko Haram, Al-Shabaab, and other groups.
- **Western Europe** contributes a visible but comparatively small layer throughout.


---
## Section 5 — Diagnostic Analysis

**Purpose:** Explain *why* patterns exist. Investigate statistical associations, test hypotheses, and identify the variables most strongly linked to terrorism frequency.

**Methods used:**
- Pearson & Spearman correlation matrices
- Pair-plot scatter analysis
- Multi-model regression (Linear, Polynomial, Gradient Boosting)
- Hypothesis testing (ANOVA, Kruskal-Wallis, Chi-square)
- Root cause / driver ranking (group summaries + linear coefficient ranking)


### 5.1  Correlation Analysis

**What this analysis does:** Computes both Pearson (linear) and Spearman (rank-based, robust to outliers) correlation matrices for the six core numeric variables. Produces heatmaps and a pair-plot.

**Question it answers:** Which numeric variables move together with `Num_Attacks`? Does wealth or population predict terrorism frequency?


In [ ]:
# ── 5.1 Pearson & Spearman Correlations ──────────────────────────
NUM_COLS = ["Num_Attacks", "Fatalities", "Injuries",
            "GDP_per_capita", "Population", "Conflict_Level"]

corr_data = master[NUM_COLS].dropna()
pearson  = corr_data.corr(method="pearson")
spearman = corr_data.corr(method="spearman")

print("=== Pearson Correlation Matrix ===")
print(pearson.round(3).to_string())

print("\n=== Spearman Correlation Matrix ===")
print(spearman.round(3).to_string())

# Top pairs
def top_off_diag(corr, n=6):
    mask = np.tril(np.ones(corr.shape, dtype=bool))
    upper = corr.where(~pd.DataFrame(mask, index=corr.index, columns=corr.columns))
    return upper.stack().abs().sort_values(ascending=False).head(n)

print("\n=== Strongest Pearson Pairs (|r|) ===")
print(top_off_diag(pearson).round(3).to_string())


In [ ]:
# ── 5.1 Correlation Heatmaps ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(pearson,  ax=axes[0], annot=True, fmt=".2f", square=True,
            cmap="coolwarm", center=0,
            cbar_kws={"label": "r"})
axes[0].set_title("Pearson Correlation", fontsize=13)

sns.heatmap(spearman, ax=axes[1], annot=True, fmt=".2f", square=True,
            cmap="coolwarm", center=0,
            cbar_kws={"label": "ρ"})
axes[1].set_title("Spearman Correlation", fontsize=13)

fig.suptitle("Correlation Matrices — Terrorism & Socioeconomic Variables", fontsize=14)
fig.tight_layout()
plt.show()


**Output Interpretation:**

- `Num_Attacks` is **very strongly correlated** with `Fatalities` (r = 0.86) and `Injuries` (r = 0.79) — more attacks produce more casualties, unsurprisingly.
- `Conflict_Level` shows a **moderate positive correlation** with `Num_Attacks` (r ≈ 0.36) — country-years with active armed conflict tend to also have higher terrorism activity, consistent with conflict-terrorism co-occurrence theory.
- `GDP_per_capita` shows a **weak negative correlation** with `Num_Attacks` (r ≈ −0.065) — wealthier countries tend to have slightly fewer attacks, but the relationship is not strong, suggesting poverty alone does not predict terrorism.
- `Population` has a **weak positive correlation** — larger countries have more attacks in raw terms, but this is a scale effect.
- The Spearman and Pearson matrices are very similar, indicating the correlations are not driven purely by extreme outliers.


### 5.2  Regression Analysis

**What this analysis does:** Fits five regression models of increasing complexity to predict `Num_Attacks` from socioeconomic and conflict features:
1. Linear baseline (5 features)
2. Linear + Year
3. Polynomial (degree 2) + scaling
4. HistGradientBoosting (tree-based, handles non-linearity)
5. Same boosting model with log1p target transformation

A fixed 80/20 train-test split (random_state=42) is used for all models.

**Question it answers:** How much of the variation in terrorism activity can be explained by observable country-level factors?


In [ ]:
# ── 5.2 Regression Analysis ──────────────────────────────────────
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import FunctionTransformer

TARGET = "Num_Attacks"
FEATURES_BASE = ["GDP_per_capita", "Population", "Conflict_Level", "Fatalities", "Injuries"]
FEATURES_YEAR = FEATURES_BASE + ["Year"]
CAT_COLS = ["Region", "Attack_Type_Mode", "Target_Type_Mode"]
NUM_EXTRA = ["Conflict", "Conflict_Intensity_deaths"]

all_feat_cols = list(dict.fromkeys(
    [TARGET] + FEATURES_YEAR + CAT_COLS + NUM_EXTRA
))
reg_df = master[all_feat_cols].dropna()

X = reg_df.drop(columns=[TARGET])
y = reg_df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def rmse(yt, yp): return float(np.sqrt(mean_squared_error(yt, yp)))

results = []

# Model 1: Linear Baseline (5 features)
m1 = Pipeline([("scaler", StandardScaler()),
               ("lr", LinearRegression())])
m1.fit(X_train[FEATURES_BASE], y_train)
r1_test = rmse(y_test, m1.predict(X_test[FEATURES_BASE]))
r1_r2   = r2_score(y_test, m1.predict(X_test[FEATURES_BASE]))
results.append({"Model": "Linear (5 features)", "RMSE_test": r1_test, "R2_test": r1_r2})

# Model 2: Linear + Year
m2 = Pipeline([("scaler", StandardScaler()),
               ("lr", LinearRegression())])
m2.fit(X_train[FEATURES_YEAR], y_train)
r2_test = rmse(y_test, m2.predict(X_test[FEATURES_YEAR]))
r2_r2   = r2_score(y_test, m2.predict(X_test[FEATURES_YEAR]))
results.append({"Model": "Linear + Year", "RMSE_test": r2_test, "R2_test": r2_r2})

# Model 3: Polynomial degree-2
m3 = Pipeline([
    ("scaler", StandardScaler()),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("lr", LinearRegression())
])
m3.fit(X_train[FEATURES_YEAR], y_train)
r3_test = rmse(y_test, m3.predict(X_test[FEATURES_YEAR]))
r3_r2   = r2_score(y_test, m3.predict(X_test[FEATURES_YEAR]))
results.append({"Model": "Polynomial deg-2", "RMSE_test": r3_test, "R2_test": r3_r2})

# Model 4: HistGradientBoosting (numeric + categorical)
num_pipe = Pipeline([("s", StandardScaler())])
cat_pipe = Pipeline([("ohe", OneHotEncoder(drop="first", sparse_output=False,
                                            handle_unknown="ignore"))])
pre4 = ColumnTransformer([
    ("num", num_pipe, FEATURES_YEAR + NUM_EXTRA),
    ("cat", cat_pipe, CAT_COLS)
])
m4 = Pipeline([("pre", pre4),
               ("hgb", HistGradientBoostingRegressor(max_iter=200, random_state=42))])
m4.fit(X_train, y_train)
r4_test = rmse(y_test, m4.predict(X_test))
r4_r2   = r2_score(y_test, m4.predict(X_test))
results.append({"Model": "HistGradientBoosting", "RMSE_test": r4_test, "R2_test": r4_r2})

# Model 5: HGB with log1p target
y_train_log = np.log1p(y_train)
m5 = Pipeline([("pre", pre4),
               ("hgb", HistGradientBoostingRegressor(max_iter=200, random_state=42))])
m5.fit(X_train, y_train_log)
pred5 = np.expm1(m5.predict(X_test))
r5_test = rmse(y_test, pred5)
r5_r2   = r2_score(y_test, pred5)
results.append({"Model": "HGB + log1p target", "RMSE_test": r5_test, "R2_test": r5_r2})

res_df = pd.DataFrame(results)
print("=== Regression Model Comparison ===")
print(res_df.round(3).to_string(index=False))


**Output Interpretation:**

- **Linear models** explain a moderate fraction of variance — R² improves when adding `Year` and then goes further with polynomial interactions.
- **HistGradientBoosting** (non-linear, handles interactions automatically) achieves substantially better RMSE and R², confirming that the relationship between covariates and attack frequency is **non-linear**.
- The **log1p target transform** further improves performance for the boosting model — the count target is right-skewed, so predicting on the log scale reduces the penalty from extreme high values.
- The overall R² remains below 1 because terrorism is partially unpredictable from aggregate socioeconomic factors alone. Attack timing and perpetrator decisions introduce irreducible uncertainty.


### 5.3  Hypothesis Testing

**What this analysis does:** Three formal statistical tests:

- **A. ANOVA + Kruskal-Wallis** — Do regions differ significantly in mean `Num_Attacks`?
- **B. Chi-square goodness-of-fit** — Is the distribution of attacks across attack types uniform?
- **C. Chi-square independence** — Are region and attack type statistically associated?

**Question it answers:** Are the observed regional and attack-type differences statistically significant, or could they arise by chance?


In [ ]:
# ── 5.3 Hypothesis Tests ─────────────────────────────────────────
hyp_df = master[["Region", "Num_Attacks", "Attack_Type_Mode"]].dropna()

# ── A. Region vs Num_Attacks ──────────────────────────────────────
groups = [g["Num_Attacks"].values for _, g in hyp_df.groupby("Region")]
f_stat, p_anova = stats.f_oneway(*groups)
h_stat, p_kw    = stats.kruskal(*groups)

print("=== A. Region vs Num_Attacks ===")
print(f"  H0: Mean Num_Attacks is equal across all 12 regions")
print(f"  H1: At least one region has a different mean")
print(f"  One-way ANOVA:    F = {f_stat:.4f},  p = {p_anova:.4e}")
print(f"  Kruskal-Wallis:   H = {h_stat:.4f},  p = {p_kw:.4e}")
if p_anova < 0.05 and p_kw < 0.05:
    print("  ✓ REJECT H0 — Significant regional differences in terrorism frequency")

# ── B. Attack type distribution ───────────────────────────────────
totals = hyp_df.groupby("Attack_Type_Mode")["Num_Attacks"].sum().sort_index()
obs  = totals.values.astype(float)
exp  = np.full(len(obs), obs.sum() / len(obs))
chi2_gof, p_gof = stats.chisquare(obs, exp)

print("\n=== B. Attack Type Goodness-of-Fit ===")
print(f"  H0: Attacks are uniformly distributed across attack types")
print(f"  H1: Some types disproportionately dominate")
print(f"  Chi-square = {chi2_gof:.4f},  df = {len(obs)-1},  p = {p_gof:.4e}")
if p_gof < 0.05:
    print("  ✓ REJECT H0 — Attack distribution is NOT uniform")
print("\n  Observed totals by type:")
for name, val in totals.items():
    print(f"    {name:45s}: {int(val):>8,}")

# ── C. Region × Attack Type independence ─────────────────────────
tab = pd.crosstab(hyp_df["Region"], hyp_df["Attack_Type_Mode"])
chi2_ct, p_ct, dof_ct, _ = stats.chi2_contingency(tab)

print("\n=== C. Region × Attack Type Independence ===")
print(f"  H0: Region and attack type are independent")
print(f"  H1: There is an association between region and attack type")
print(f"  Chi-square = {chi2_ct:.4f},  df = {dof_ct},  p = {p_ct:.4e}")
print(f"  Table shape: {tab.shape[0]} regions × {tab.shape[1]} attack types")
if p_ct < 0.05:
    print("  ✓ REJECT H0 — Region and attack type ARE statistically associated")


**Output Interpretation:**

- **All three tests** reject the null hypothesis at extremely small p-values (p << 0.001).
- **Regional differences** in terrorism frequency are statistically significant — the choice of region explains substantial variation (ANOVA F is large, and the non-parametric Kruskal-Wallis confirms it is not an outlier-driven result).
- **Attack type distribution** is far from uniform — Bombing/Explosion dominates while Hijacking is rare. This has operational implications: counter-terrorism resources should be concentrated on IED/explosive threats.
- **Region and attack type are associated** — different regions have characteristic preferred attack methods. Middle Eastern attacks skew toward bombing; kidnapping is more prevalent in Sub-Saharan Africa.


### 5.4  Root Cause Driver Analysis

**What this analysis does:** Two complementary approaches:
1. **Group summary tables** — mean/median attacks and conflict metrics by Region and Attack Type
2. **Scaled linear regression coefficients** — features (numeric + Region one-hot) are standardised and a linear model is fit; the ranked absolute coefficients indicate which variables have the strongest *linear* association with `Num_Attacks`

**Question it answers:** What are the strongest observable drivers of terrorism frequency?


In [ ]:
# ── 5.4 Root Cause — Group Tables ────────────────────────────────
root_df = master[["Num_Attacks","Fatalities","Injuries","GDP_per_capita",
                   "Population","Conflict_Level","Conflict","Year",
                   "Region","Attack_Type_Mode"]].dropna()

# Region group table
region_stats = (
    root_df.groupby("Region")
    .agg(
        n_rows        =("Num_Attacks", "count"),
        mean_attacks  =("Num_Attacks", "mean"),
        median_attacks=("Num_Attacks", "median"),
        mean_conflict =("Conflict_Level", "mean"),
        share_conflict=("Conflict", "mean"),
    )
    .sort_values("mean_attacks", ascending=False)
)
print("=== Region Group Summary ===")
print(region_stats.round(3).to_string())

# Attack type group table
type_stats = (
    root_df.groupby("Attack_Type_Mode")
    .agg(
        n_rows        =("Num_Attacks","count"),
        mean_attacks  =("Num_Attacks","mean"),
        median_attacks=("Num_Attacks","median"),
    )
    .sort_values("mean_attacks", ascending=False)
)
print("\n=== Attack Type Group Summary ===")
print(type_stats.round(3).to_string())


In [ ]:
# ── 5.4 Root Cause — Scaled Linear Coefficient Ranking ───────────
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

NUM_FEATS = ["GDP_per_capita","Population","Conflict_Level",
             "Fatalities","Injuries","Year","Conflict"]
CAT_FEATS = ["Region"]

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM_FEATS),
    ("cat", OneHotEncoder(drop="first", sparse_output=False,
                          handle_unknown="ignore"), CAT_FEATS)
])
m = Pipeline([("pre", pre), ("lr", LinearRegression())])
m.fit(root_df[NUM_FEATS + CAT_FEATS], root_df["Num_Attacks"])

feat_names = m.named_steps["pre"].get_feature_names_out()
coefs = m.named_steps["lr"].coef_
ranking = (
    pd.DataFrame({"Feature": feat_names,
                  "Coefficient": coefs,
                  "|Coefficient|": np.abs(coefs)})
    .sort_values("|Coefficient|", ascending=False)
    .reset_index(drop=True)
)
r2_full = m.score(root_df[NUM_FEATS + CAT_FEATS], root_df["Num_Attacks"])

print(f"=== Scaled Linear Model — Full-Sample R² = {r2_full:.4f} ===")
print("\nRanked Features by |Coefficient|:")
print(ranking.round(3).to_string(index=False))

# Visualise top 15
fig, ax = plt.subplots(figsize=(10, 6))
top15 = ranking.head(15).sort_values("|Coefficient|")
colors = ["#d62728" if v >= 0 else "#1f77b4" for v in top15["Coefficient"]]
ax.barh(top15["Feature"].str.replace("num__","").str.replace("cat__Region_",""),
        top15["Coefficient"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_title("Standardised Linear Regression Coefficients (Top 15 Drivers)", fontsize=13)
ax.set_xlabel("Coefficient (on standardised features)", fontsize=11)
fig.tight_layout()
plt.show()


**Output Interpretation:**

- **Fatalities** has the largest coefficient by far — but this is partly tautological (more attacks cause more deaths, and the model also sees deaths as a predictor). This confirms both variables move together.
- **South Asia region** has the strongest positive regional coefficient, confirming it as structurally different from the reference region.
- **Conflict_Level** is a significant positive driver — countries in active armed conflict have systematically more terrorism events.
- **GDP_per_capita** has a negative coefficient, consistent with the correlation analysis: wealthier countries tend to have fewer attacks.
- **Year** shows a positive coefficient, capturing the secular upward trend.
- The **group summary** confirms South Asia's exceptional mean (205 attacks/country-year vs. global mean ~48), and that Bombing/Explosion is not only the most common type but also associated with the highest mean attack counts per country-year.


---
## Section 6 — Predictive Analysis

**Purpose:** Forecast *what will happen*. Two complementary predictive models are built:

1. **Global time-series regression** — Predict total global attacks at 1-, 5-, and 10-year horizons using lag features.
2. **Panel model + risk classification** — Country-year level forecasting and binary high-risk classification using Random Forest.

**Methods used:**
- Ridge regression with lag features (autoregressive)
- Random Forest regressor (panel, multi-horizon)
- Direct multi-horizon targets (no iterative chaining)
- GroupKFold cross-validation (unseen countries)
- Binary risk classification (Logistic Regression, Decision Tree, Random Forest)
- Feature importance


### 6.1  Global Forecast — Lag-Based Ridge Regression

**What this analysis does:** Aggregates the dataset to global yearly totals, engineers a single lag-1 feature, and uses RidgeCV in a walk-forward backtest across horizons of 1, 5, and 10 years. The model then produces forward forecasts for 2018, 2022, and 2027.

**Question it answers:** How reliably can historical attack trends predict future global terrorism levels? Does accuracy degrade at longer horizons?


In [ ]:
# ── 6.1 Global Forecast — Build & Evaluate ───────────────────────
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Build global yearly series (1993 is missing in GTD — fill with 0)
yearly_all = master.groupby("Year")["Num_Attacks"].sum()
full_idx = pd.RangeIndex(int(yearly_all.index.min()), int(yearly_all.index.max()) + 1)
yearly_all = yearly_all.reindex(full_idx, fill_value=0)

# Supervised frame with lag-1
sup = pd.DataFrame({
    "year": yearly_all.index.astype(int),
    "y":    yearly_all.values.astype(float),
    "lag1": yearly_all.values.astype(float)
})
sup["lag1"] = sup["y"].shift(1)
sup = sup.dropna().reset_index(drop=True)

def ridge_model():
    return Pipeline([("s", StandardScaler()),
                     ("r", RidgeCV(alphas=np.logspace(-3, 6, 30)))])

def iterative_predict(model, start_lag, start_year, steps):
    lag = float(start_lag)
    for k in range(1, steps + 1):
        X = pd.DataFrame({"lag1":[lag], "year":[start_year+k]})
        lag = float(model.predict(X)[0])
    return lag

# Back-test RMSE by horizon
ANCHORS  = list(range(1985, 2010))
MIN_YEAR = 1975
backtest_res = []
for hz in [1, 5, 10]:
    errs = []
    for anchor in ANCHORS:
        train = sup[(sup["year"] < anchor) & (sup["year"] >= MIN_YEAR)]
        if len(train) < 5: continue
        fut = sup.loc[sup["year"] == anchor + hz, "y"]
        if fut.empty: continue
        m = ridge_model()
        m.fit(train[["lag1","year"]], train["y"])
        lag0 = float(sup.loc[sup["year"] == anchor, "y"].values[0])
        pred = iterative_predict(m, lag0, anchor, hz)
        errs.append((pred - float(fut.values[0]))**2)
    rmse_hz = float(np.sqrt(np.mean(errs))) if errs else float("nan")
    backtest_res.append({"Horizon (years)": hz, "Backtest RMSE": round(rmse_hz, 1),
                         "Anchors used": len(errs)})

rmse_df = pd.DataFrame(backtest_res)
print("=== Back-test RMSE by Forecast Horizon ===")
print(rmse_df.to_string(index=False))

# Forward forecasts
last_year = int(sup["year"].max())
last_lag  = float(sup.loc[sup["year"] == last_year, "y"].values[0])

m_full = ridge_model()
m_full.fit(sup[["lag1","year"]], sup["y"])

forecasts = []
for hz in [1, 5, 10]:
    pred = iterative_predict(m_full, last_lag, last_year, hz)
    forecasts.append({"Forecast Year": last_year + hz, "Horizon": hz,
                      "Predicted Total Attacks": round(pred, 0)})
fcast_df = pd.DataFrame(forecasts)
print("\n=== Forward Forecasts (from {}–{}) ===".format(last_year, last_year+10))
print(fcast_df.to_string(index=False))


In [ ]:
# ── 6.1 Forecast Visualisation ───────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(sup["year"], sup["y"], linewidth=2, label="Historical total attacks")

fcast_years  = [last_year + hz for hz in [1, 5, 10]]
fcast_values = [float(fcast_df.loc[fcast_df["Horizon"]==hz,
                    "Predicted Total Attacks"].values[0]) for hz in [1,5,10]]

ax.scatter(fcast_years, fcast_values, s=100, zorder=5, label="Forecasts", marker="^")
for yr, val in zip(fcast_years, fcast_values):
    ax.annotate(f"{yr}: {val:,.0f}", (yr, val),
                textcoords="offset points", xytext=(8, 5), fontsize=9)
ax.axvline(last_year, linestyle="--", linewidth=1, alpha=0.6, label="Last observed year")
ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Total Attacks", fontsize=12)
ax.set_title("Global Terrorism Forecast — Lag-1 Ridge Regression", fontsize=13)
ax.legend()
fig.tight_layout()
plt.show()


**Output Interpretation:**

- **1-year RMSE** (~2,229) is the smallest, confirming forecasts degrade at longer horizons — a universal property of autoregressive models.
- At **5 years**, RMSE roughly doubles (~4,801), and at **10 years** it rises to ~7,513.
- The **forward forecasts** project global totals in the range of 10,800–11,700 for 2018, 2022, and 2027 — a relative plateau from the 2017 observed level, suggesting the model expects a levelling-off rather than continued exponential growth.
- Note that this is a **global aggregate model** — it does not capture country-level heterogeneity. The panel model (Section 6.2) addresses this limitation.


### 6.2  Panel Model — Country-Year Random Forest Forecast

**What this analysis does:** Constructs a **balanced country-year panel** (every country × every year from dataset min to max, filling missing years with 0), engineers lag and rolling features *without data leakage*, and trains Random Forest and Ridge regressors to predict attacks at 1-, 5-, and 10-year direct horizons. GroupKFold (by country) provides a rigorous out-of-sample evaluation.

**Question it answers:** Can we predict a specific country's attack count next year, 5 years out, and 10 years out? How does model performance change across horizons?

**Features engineered (no same-year leakage):**
- `num_attacks_lag1`, `num_attacks_lag2` — prior-year attack counts
- `num_attacks_roll3` — 3-year rolling mean of *past* attacks only (shift(1) first)
- `log_gdp_lag1`, `log_pop_lag1` — log1p of prior-year GDP and population
- `conflict_level_lag1`, `conflict_lag1` — prior-year conflict indicators
- `Year` — calendar year (trend)
- `Region` — one-hot encoded


In [ ]:
# ── 6.2 Panel Forecast Results ───────────────────────────────────
panel_rmse_path = ANALYSIS / "predictive" / "panelModel" / "outputs" / "direct_horizon_groupkfold_rmse.csv"
if panel_rmse_path.exists():
    panel_rmse = pd.read_csv(panel_rmse_path)
    print("=== Panel Model — GroupKFold RMSE (unseen countries) ===")
    print(panel_rmse.round(3).to_string(index=False))
    print("\nInterpretation:")
    print("  - Lower RMSE = better; RF and Ridge are compared at each horizon")
    print("  - std shows variability across folds")
    print("  - t+10 is hardest — RMSE is highest because 10-year noise accumulates")
else:
    # Compute inline if outputs not found
    print("Panel model outputs not found — computing inline summary...")
    print("  (Run analysis/predictive/panelModel/panel_forecast_and_risk.py to generate full outputs)")


In [ ]:
# ── 6.2 Feature Importance — Panel Model ─────────────────────────
fi_path = ANALYSIS / "predictive" / "panelModel" / "outputs" / "feature_importance_h1.png"

# Display pre-computed feature importance (h=1 horizon)
h1_rmse_path = ANALYSIS / "predictive" / "panelModel" / "outputs" / "country_rmse_h1_test.csv"
if h1_rmse_path.exists():
    h1_rmse = pd.read_csv(h1_rmse_path)
    rmse_col = "rmse_rf" if "rmse_rf" in h1_rmse.columns else "rmse_h1"

    print("=== Country-Level RMSE (Horizon 1, Temporal Test Set) — Top & Bottom 10 ===")
    print(f"  Using RMSE column: {rmse_col}")
    print("  Best-predicted (lowest RMSE):")
    print(h1_rmse.nsmallest(10, rmse_col).to_string(index=False))
    print("  Hardest to predict (highest RMSE):")
    print(h1_rmse.nlargest(10, rmse_col).to_string(index=False))

print("\n(Feature importance plot saved to:", fi_path, ")")


**Output Interpretation:**

- **GroupKFold RMSE** is the most reliable metric: at horizon 1, RF achieves a mean RMSE of ~63 attacks/country-year, while Ridge achieves ~58. Both degrade progressively at horizons 5 (~85 attacks) and 10 (~101 attacks).
- The **`attacks_roll3` (rolling mean)** is the top feature, confirming that recent momentum is the best predictor of near-term activity — regions/countries with persistent high activity tend to remain high.
- **`fatalities_lag1`** is also highly important, reflecting the intensity of past attacks as a signal of continued conflict.
- **High-volume, unstable countries** (Iraq, Afghanistan during peak periods) are the hardest to predict because they experience large inter-year swings tied to geopolitical events not captured in the feature set.


### 6.3  Risk Classification — Region and Country Level

**What this analysis does:** Labels each region-year (and country-year) as **High Risk** (top 25% by risk score = attacks + 0.5 × fatalities) or Not High Risk. Trains Logistic Regression, Decision Tree, Random Forest, and a Dummy baseline under a **temporal train/test split** (train ≤ 2010, test > 2010). Evaluates accuracy, F1 (high-risk class), ROC-AUC, and repeats under GroupKFold.

**Question it answers:** Can we classify which regions will be high-risk in future time periods? Which features drive high-risk prediction?


In [ ]:
# ── 6.3 Classification — Temporal Results ────────────────────────
clf_path = ANALYSIS / "predictive" / "classification" / "outputs" / "classification_temporal_metrics_region_vs_country.csv"
if clf_path.exists():
    clf = pd.read_csv(clf_path)
    region_clf = clf[clf["level"] == "region"].copy()
    print("=== Region-Level Classification — Temporal Test (Year > 2010) ===")
    cols_show = ["model","accuracy","precision_high_risk","recall_high_risk",
                 "f1_high_risk","roc_auc"]
    print(region_clf[cols_show].round(3).to_string(index=False))


In [ ]:
# ── 6.3 GroupKFold Results ────────────────────────────────────────
gkf_path = ANALYSIS / "predictive" / "classification" / "outputs" / "groupkfold_metrics_region.csv"
if gkf_path.exists():
    gkf = pd.read_csv(gkf_path)
    print("=== GroupKFold (Region-held-out) — Mean Accuracy & F1 ===")
    print(gkf.round(3).to_string(index=False))


In [ ]:
# ── 6.3 Feature Importance — RF Classifier ───────────────────────
fi_clf_path = ANALYSIS / "predictive" / "classification" / "outputs" / "feature_importance_rf_region_top10.csv"
if fi_clf_path.exists():
    fi_clf = pd.read_csv(fi_clf_path)
    print("=== RF Feature Importance (Region Risk Classification, Top 10) ===")
    print(fi_clf.round(4).to_string(index=False))

    fig, ax = plt.subplots(figsize=(9, 5))
    fi_clf_sorted = fi_clf.sort_values("importance")
    ax.barh(fi_clf_sorted["feature"].str.replace("num__","").str.replace("cat__Region_",""),
            fi_clf_sorted["importance"],
            color=plt.cm.Blues(np.linspace(0.4, 0.9, len(fi_clf_sorted))))
    ax.set_title("Random Forest Feature Importance — Region Risk Classification", fontsize=13)
    ax.set_xlabel("Importance (Mean Decrease in Impurity)", fontsize=11)
    fig.tight_layout()
    plt.show()


In [ ]:
# ── 6.3 Future High-Risk Regions ─────────────────────────────────
hr_path = ANALYSIS / "predictive" / "classification" / "outputs" / "regions_ranked_future_high_risk.csv"
if hr_path.exists():
    hr = pd.read_csv(hr_path)
    print("=== Regions Ranked by Future High-Risk Probability (RF) ===")
    print(hr.round(4).to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 6))
    hr_s = hr.sort_values("mean_proba_high_rf", ascending=True)
    colors_bar = ["#d62728" if p >= 0.5 else "#2196F3"
                  for p in hr_s["mean_proba_high_rf"]]
    bars = ax.barh(hr_s["Region"], hr_s["mean_proba_high_rf"], color=colors_bar)
    ax.axvline(0.5, linestyle="--", linewidth=1.2, alpha=0.8, label="50% threshold")
    ax.set_xlim(0, 1.1)
    ax.set_xlabel("Mean Predicted Probability of High-Risk", fontsize=12)
    ax.set_title("Future High-Risk Probability by Region (Random Forest)", fontsize=13)
    for bar, val in zip(bars, hr_s["mean_proba_high_rf"]):
        ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
                f"{val:.1%}", va="center", fontsize=9)
    ax.legend()
    fig.tight_layout()
    plt.show()


**Output Interpretation:**

- **Random Forest** achieves 95% accuracy and F1 = 0.93 (high-risk class) on the temporal test set — substantially better than the baseline (majority-class predictor: 65% accuracy, F1 = 0).
- **GroupKFold** (held-out regions) reduces mean accuracy to ~93% and F1 to ~0.82 — still strong, confirming the model generalises to unseen regions.
- **`fatalities_lag1`** and `attacks_roll3` dominate feature importance, showing that recent history is the most reliable indicator of sustained high risk. Calendar `Year` also matters, reflecting structural trend.
- **Future high-risk ranking:** South Asia (99.9%), MENA (99.2%), Sub-Saharan Africa (98.2%), Southeast Asia (82.7%) are nearly certain to remain high-risk based on historical patterns. Western Europe (19%) and Australasia (0.03%) are very unlikely.


---
## Section 7 — Prescriptive Analysis

**Purpose:** Translate statistical insights into **actionable decisions**. Prescriptive analysis closes the analytics loop by answering: *Given what we know, what should be done?*

Two components:

1. **Rule-Based Intervention Engine** — IF-THEN rules derived from diagnostic and predictive findings that fire specific policy recommendations
2. **Resource Allocation Optimisation** — Linear programming to optimally distribute a security budget across 12 world regions, minimising total expected harm

**Question it answers:** How should counter-terrorism resources be allocated? Which countries and regions need what kinds of interventions?


### 7.1  Rule-Based Intervention Engine

**What this analysis does:** Applies 8 structured IF-THEN rules to the most recent observation year per country (≥ 2005). Rules are derived from the diagnostic findings and predictive risk scores. Each triggered rule emits a structured recommendation with a priority level (CRITICAL, HIGH, MEDIUM) and an action category.

| Rule ID | Trigger Condition | Priority | Action Category |
|---------|------------------|----------|----------------|
| R01 | Bombing dominant + high-risk region + ≥100 attacks | CRITICAL | Counter-IED Capability |
| R02 | Attacks >20% above 3-year rolling average | HIGH | Early Warning / Surge Response |
| R03 | Conflict Level ≥ 2 AND low-GDP country | HIGH | Stabilisation / Development Aid |
| R04 | Private Citizens dominant target + ≥10 attacks | HIGH | Protective Security |
| R05 | Armed Assault dominant + ≥20 attacks | MEDIUM | Armed Forces / Border Security |
| R06 | Kidnapping dominant attack type | MEDIUM | Hostage / Ransom Policy |
| R07 | Medium risk region + ≥5 attacks | MEDIUM | Intelligence / Monitoring |
| R08 | Diplomatic/Government target spike + ≥15 attacks | HIGH | Diplomatic Security |

**Question it answers:** Which specific countries need targeted intervention right now, and what type of intervention is most appropriate?


In [ ]:
# ── 7.1 Load Rule Engine Results ─────────────────────────────────
rules_path = ANALYSIS / "prescriptive" / "rule_engine_fired.csv"
if rules_path.exists():
    rules = pd.read_csv(rules_path)
    print("=== Rule Engine — Summary ===")
    print(f"  Total rule triggers:   {len(rules):,}")
    print(f"  Countries affected:    {rules['Country'].nunique()}")
    print(f"  Regions affected:      {rules['Region'].nunique()}")
    print()
    
    print("Triggers by priority:")
    print(rules["priority"].value_counts().to_string())
    
    print("\nTriggers by rule:")
    print(rules.groupby(["rule_id","rule_name","priority"])
          .size().reset_index(name="triggers").to_string(index=False))


In [ ]:
# ── 7.1 Visualise — Triggers by Region and Priority ──────────────
if rules_path.exists():
    # Triggers by region
    region_triggers = rules.groupby(["Region","priority"]).size().unstack(fill_value=0)
    priority_order  = ["CRITICAL","HIGH","MEDIUM"]
    for p in priority_order:
        if p not in region_triggers.columns:
            region_triggers[p] = 0
    region_triggers = region_triggers[priority_order]
    region_triggers["Total"] = region_triggers.sum(axis=1)
    region_triggers = region_triggers.sort_values("Total", ascending=False)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: triggers by region
    region_triggers[priority_order].plot(kind="barh", stacked=True,
                                         ax=axes[0], colormap="Reds")
    axes[0].set_title("Rule Engine Triggers by Region & Priority", fontsize=13)
    axes[0].set_xlabel("Number of Triggers")
    axes[0].legend(title="Priority", loc="lower right")

    # Right: triggers by rule
    rule_summary = rules.groupby("rule_id").size().sort_values(ascending=True)
    axes[1].barh(rule_summary.index, rule_summary.values,
                 color=plt.cm.Oranges(np.linspace(0.4, 0.9, len(rule_summary))))
    axes[1].set_title("Rule Engine Triggers by Rule ID", fontsize=13)
    axes[1].set_xlabel("Number of Triggers")

    fig.tight_layout()
    plt.show()

    print("\n=== Sample Triggered Recommendations (first 10) ===")
    cols_show = ["Country","Region","Year","priority","rule_name","recommendation"]
    sample = rules[cols_show].head(10)
    for _, row in sample.iterrows():
        print(f"  [{row['priority']:8s}] {row['Country']:30s} ({row['Region']}) — {row['rule_name']}")
        print(f"             → {row['recommendation'][:110]}...")
        print()


**Output Interpretation:**

- **87 total rule triggers** fired across 53 countries and 8 regions.
- **9 CRITICAL triggers** were issued — all for South Asian and MENA countries with sustained high-volume bombing activity (Rule R01).
- **R03 (Conflict + Low GDP)** is the most widely fired rule, reflecting that fragile-state conditions (poverty × active conflict) are pervasive across Sub-Saharan Africa and parts of Asia. This aligns with the diagnostic finding that `Conflict_Level` is a significant predictor.
- **Sub-Saharan Africa** generates the most HIGH/MEDIUM triggers, reflecting its growing share of terrorism in the 2010s despite historically lower intensity.
- The rule engine provides **decision-support outputs** that can be reviewed by security analysts — not autonomous decisions.


### 7.2  Resource Allocation Optimisation

**What this analysis does:** Formulates a **nonlinear optimisation problem** using SciPy SLSQP to allocate a fixed security budget (100%) across 12 world regions to minimise total expected terrorism harm.

**Mathematical formulation:**
- **Objective:** Minimise Σ `harm_i × (1 − effectiveness(x_i))`
- **Effectiveness model:** `min(0.70, 0.55 × √(x_i / equal_share))` — diminishing returns
- **Constraints:** `Σ x_i = 100` (full budget), `x_i ≥ 2` (minimum floor), `x_i ≤ 40` (concentration cap)
- **Harm measure:** Region's mean annual `risk_score = attacks + 0.5 × fatalities` (training period ≤ 2010)

**Question it answers:** Given limited counter-terrorism resources, how should they be distributed across regions to minimise overall harm? What is the expected harm reduction from optimal allocation?


In [ ]:
# ── 7.2 Resource Allocation Results ──────────────────────────────
alloc_path = ANALYSIS / "prescriptive" / "resource_allocation.csv"
if alloc_path.exists():
    alloc = pd.read_csv(alloc_path)
    alloc = alloc.sort_values("Baseline_Risk_Score", ascending=False)
    
    print("=== Optimal Resource Allocation Results ===")
    print(alloc.round(2).to_string(index=False))
    
    total_baseline = alloc["Baseline_Risk_Score"].sum()
    total_residual = alloc["Harm_After"].sum()
    overall_reduction = (1 - total_residual / total_baseline) * 100
    print(f"\n  Total baseline risk:    {total_baseline:.1f}")
    print(f"  Total residual risk:    {total_residual:.1f}")
    print(f"  Overall harm reduction: {overall_reduction:.1f}%")


In [ ]:
# ── 7.2 Allocation Visualisation ─────────────────────────────────
if alloc_path.exists():
    alloc_s = alloc.sort_values("Budget_Allocation_pct", ascending=True)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))

    # Left: budget allocation
    bar_colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(alloc_s)))
    axes[0].barh(alloc_s["Region"], alloc_s["Budget_Allocation_pct"],
                 color=bar_colors)
    axes[0].axvline(100/12, linestyle="--", alpha=0.7, label="Equal split (8.3%)")
    axes[0].set_title("Optimal Budget Allocation per Region (%)", fontsize=13)
    axes[0].set_xlabel("% of Total Security Budget")
    axes[0].legend()

    # Right: baseline vs residual risk
    x_pos = np.arange(len(alloc_s))
    width = 0.35
    axes[1].barh(x_pos + width/2, alloc_s["Baseline_Risk_Score"],
                 width, label="Baseline Risk", color="#d62728", alpha=0.8)
    axes[1].barh(x_pos - width/2, alloc_s["Harm_After"],
                 width, label="Residual Risk (after allocation)", color="#2196F3", alpha=0.8)
    axes[1].set_yticks(x_pos)
    axes[1].set_yticklabels(alloc_s["Region"])
    axes[1].set_title("Baseline vs Residual Risk per Region", fontsize=13)
    axes[1].set_xlabel("Risk Score (attacks + 0.5 × fatalities)")
    axes[1].legend()

    fig.suptitle("Optimal Counter-Terrorism Resource Allocation", fontsize=14)
    fig.tight_layout()
    plt.show()


**Output Interpretation:**

- The **overall expected harm reduction is ~65%** under optimal allocation — compared to ~27% under equal distribution. Optimal allocation nearly 2.5× more effective than a naive equal-share approach.
- **South Asia, MENA, Central America & Caribbean, South America, and Southeast Asia** receive the maximum efficient allocation (~13.5% each) — their high baseline risk justifies disproportionate resource concentration.
- **Eastern Europe, Central Asia, and Australasia** receive the minimum floor allocation (2%) because marginal harm-reduction returns are small relative to their low baseline risk scores.
- **South Asia retains the highest residual risk** even after optimal allocation — indicating that resource allocation alone is insufficient. Structural interventions (R03: fragile-state development aid) are an essential complement.
- The **diminishing-returns effectiveness model** prevents over-concentration: beyond a certain budget share, marginal security gains decrease rapidly, so spreading resources across multiple high-risk regions is more efficient than pouring everything into one.


---
## Section 8 — Conclusions

### Answering the Research Question

> **How can historical terrorism data be analyzed to uncover patterns in attack types, locations, targets, and perpetrators, and how effectively can these insights be used to assess terrorism risk across regions and time periods?**

This project demonstrated a complete data science pipeline from raw data ingestion to actionable prescriptive recommendations:

---

### Key Findings

| Analysis Type | Finding |
|--------------|---------|
| **Descriptive** | Terrorism is geographically concentrated — South Asia (mean 205 attacks/country-year), MENA (~94), and Southeast Asia (~57) account for the majority of global incidents. Bombing/Explosion is the dominant attack method (~45% of attacks). |
| **Descriptive** | Global terrorism rose sharply from 2003–2016, with peak years driven by the Iraq insurgency, Syrian Civil War, and Taliban/ISIS activity. The linear trend is positive (+60 attacks/year). |
| **Diagnostic** | Conflict level (r ≈ 0.36), fatalities, and injuries are most strongly correlated with attack frequency. Wealthier nations (higher GDP) have modestly fewer attacks. Correlations are non-linear — tree-based models outperform linear ones. |
| **Diagnostic** | All three hypothesis tests (ANOVA, chi-square) reject the null at p < 0.001: regions differ significantly, attack types are non-uniformly distributed, and region and attack type are associated. |
| **Predictive** | The lag-1 Ridge regression achieves RMSE = 2,229 (1-yr horizon), degrading to 7,513 at 10-year horizon. Panel RF achieves GroupKFold RMSE ~63 (1-yr, unseen countries). |
| **Predictive** | RF classification achieves 95% accuracy and F1 = 0.93 for high-risk region-years. South Asia (99.9%), MENA (99.2%), Sub-Saharan Africa (98.2%) are ranked as near-certain future high-risk regions. |
| **Prescriptive** | 87 rule triggers across 53 countries; R03 (Conflict + Low GDP) is most prevalent. Optimal resource allocation reduces total harm by ~65% vs ~27% under equal distribution. South Asia requires both resource allocation AND structural intervention. |

---

### Answering the Motivating Questions

| Question | Answer |
|----------|--------|
| Where are threats most concentrated? | **South Asia and MENA** — especially Afghanistan, Iraq, Pakistan. |
| How do patterns change over time? | **Sustained long-term increase** until 2016; regional leadership shifted from South Asia to MENA during 2013–2016. |
| What can past data reveal about future risk? | **Lag and rolling features** (recent momentum) are the best predictors. 1-year forecasts are reasonably reliable; 10-year forecasts are rough directional estimates only. |
| Which patterns are hidden in raw data? | **Conflict co-occurrence** with terrorism; **attack-type–region associations**; the fragile-state (poverty × conflict) nexus as a structural driver. |
| Can data science identify high-risk regions? | **Yes** — the RF classifier achieves 95% accuracy with F1 = 0.93; future risk ranking is consistent with expert assessments. |
